# 10 — Dual-Stream Face + Mouth V3 Full Training

เวอร์ชันแก้จาก V2 หลังพบ CTC blank collapse (`CER≈1.0`, `blank≈100%`).

- Face + Mouth dual-stream
- Mouth residual fusion
- Anti-blank regularization
- Temporary blank-logit penalty
- Fast per-clip cache
- CTC compatibility filter
- Progress + ETA
- Early stopping / best / last / resume
- Final test

In [ ]:
import sys, json, random, re, time, unicodedata
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('='*80)
print('ENVIRONMENT')
print('='*80)
print('Python :', sys.executable)
print('Version:', sys.version.split()[0])
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('CUDA ไม่พร้อมใช้งาน')
DEVICE=torch.device('cuda')
print('GPU    :', torch.cuda.get_device_name(0))
print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),'GB')
torch.backends.cudnn.benchmark=True
try: torch.set_float32_matmul_precision('high')
except Exception: pass

In [ ]:
PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID='video001'
SPLIT_ROOT=PROJECT_ROOT/'dataset'/'splits'/VIDEO_ID
TRAIN_CSV=SPLIT_ROOT/'train.csv'; VAL_CSV=SPLIT_ROOT/'val.csv'; TEST_CSV=SPLIT_ROOT/'test.csv'
FACE_CACHE_ROOT=PROJECT_ROOT/'dataset'/'face_crops_v2'/VIDEO_ID
FAST_CACHE_ROOT=PROJECT_ROOT/'dataset'/'dualstream_fast_cache_v3'/VIDEO_ID
MODEL_ROOT=PROJECT_ROOT/'models'/'lipreading_dualstream_v3'
FAST_CACHE_ROOT.mkdir(parents=True,exist_ok=True); MODEL_ROOT.mkdir(parents=True,exist_ok=True)
FACE_SIZE=112; MOUTH_SIZE=96; MAX_FRAMES=140
BATCH_SIZE=2; NUM_WORKERS=0; GRAD_ACCUM_STEPS=2
FEATURE_DIM=128; GRU_HIDDEN=192; GRU_LAYERS=2; DROPOUT=0.10
LR=2e-4; WEIGHT_DECAY=1e-4; EPOCHS=30; PATIENCE=8; GRAD_CLIP=5.0; SEED=42
BLANK_REG_WEIGHT=0.35; BLANK_MARGIN=0.10
INITIAL_BLANK_LOGIT_PENALTY=1.25; BLANK_PENALTY_DECAY_EPOCHS=10
RESUME_IF_AVAILABLE=True
for p in [TRAIN_CSV,VAL_CSV,TEST_CSV,FACE_CACHE_ROOT]: print(p,'->',p.exists())
print('FAST_CACHE_ROOT:',FAST_CACHE_ROOT); print('MODEL_ROOT:',MODEL_ROOT)

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('Seed ready:',SEED)

In [ ]:
train_df=pd.read_csv(TRAIN_CSV); val_df=pd.read_csv(VAL_CSV); test_df=pd.read_csv(TEST_CSV)
required={'clip_id','speaker_id','text','frame_paths'}
for name,df in [('Train',train_df),('Val',val_df),('Test',test_df)]:
    missing=required-set(df.columns)
    if missing: raise KeyError(f'{name} CSV ขาด columns: {sorted(missing)}')
    print(f'{name}: {len(df)} clips')
display(train_df.head(3))

In [ ]:
def normalize_text(text):
    text=unicodedata.normalize('NFC',str(text)); return ' '.join(text.split()).strip()
for df in (train_df,val_df,test_df): df['text']=df['text'].fillna('').astype(str).map(normalize_text)
BLANK_TOKEN='<blank>'; UNK_TOKEN='<unk>'
charset=sorted(set(''.join(train_df['text'].tolist()))); charset=[c for c in charset if c not in ['\n','\r','\t']]
itos=[BLANK_TOKEN,UNK_TOKEN]+charset; stoi={tok:i for i,tok in enumerate(itos)}
BLANK_ID=stoi[BLANK_TOKEN]; UNK_ID=stoi[UNK_TOKEN]; VOCAB_SIZE=len(itos)
def encode_text(text): return [stoi.get(ch,UNK_ID) for ch in normalize_text(text)]
def decode_target(ids):
    out=[]
    for idx in ids:
        idx=int(idx)
        if idx==BLANK_ID: continue
        tok=itos[idx]; out.append('�' if tok==UNK_TOKEN else tok)
    return ''.join(out)
def ctc_greedy_decode(logits,lengths):
    pred_ids=logits.argmax(dim=-1); results=[]
    for b in range(pred_ids.size(0)):
        seq=pred_ids[b,:int(lengths[b])].tolist(); collapsed=[]; prev=None
        for idx in seq:
            if idx!=prev and idx!=BLANK_ID: collapsed.append(idx)
            prev=idx
        results.append(decode_target(collapsed))
    return results
with open(MODEL_ROOT/'charset.json','w',encoding='utf-8') as f: json.dump({'itos':itos,'blank_id':BLANK_ID,'unk_id':UNK_ID},f,ensure_ascii=False,indent=2)
print('VOCAB_SIZE:',VOCAB_SIZE,'BLANK_ID:',BLANK_ID)

In [ ]:
def min_ctc_steps(text):
    ids=encode_text(text); return len(ids)+sum(ids[i]==ids[i-1] for i in range(1,len(ids)))
def available_steps(row):
    paths=[p for p in str(row['frame_paths']).split('|') if p.strip()]; return min(len(paths),MAX_FRAMES)
def filter_ctc_valid(df,name):
    keep=[]; bad=[]
    for idx,row in df.iterrows():
        steps=available_steps(row); need=min_ctc_steps(row['text'])
        if steps>=need and len(encode_text(row['text']))>0: keep.append(idx)
        else: bad.append((row['clip_id'],steps,need))
    out=df.loc[keep].reset_index(drop=True); print(f'{name}: {len(df)} -> {len(out)} | removed={len(bad)}')
    if bad: print('removed examples:',bad[:5])
    return out
train_df=filter_ctc_valid(train_df,'Train'); val_df=filter_ctc_valid(val_df,'Val'); test_df=filter_ctc_valid(test_df,'Test')

In [ ]:
frame_re=re.compile(r'frame_(\d+)',re.IGNORECASE)
def resolve_path(p):
    p=Path(str(p)); return p if p.is_absolute() else PROJECT_ROOT/p
def select_paths(paths,max_frames):
    if len(paths)<=max_frames:return paths
    idx=np.linspace(0,len(paths)-1,max_frames).astype(int); return [paths[i] for i in idx]
def face_cache_path(speaker_id,frame_index):
    return FACE_CACHE_ROOT/f'speaker_{int(speaker_id):03d}'/f'frame_{int(frame_index):08d}.jpg'
def clip_cache_path(clip_id):
    safe=re.sub(r'[^A-Za-z0-9_.-]+','_',str(clip_id)); return FAST_CACHE_ROOT/f'{safe}.npz'
def build_one_cache(row):
    dst=clip_cache_path(row['clip_id'])
    if dst.exists(): return 'skip'
    spk=int(row['speaker_id']); paths=[p for p in str(row['frame_paths']).split('|') if p.strip()]; paths=select_paths(paths,MAX_FRAMES)
    faces=[]; mouths=[]; indices=[]
    for p in paths:
        mp=resolve_path(p); m=frame_re.search(mp.stem)
        if m is None: continue
        fi=int(m.group(1)); fp=face_cache_path(spk,fi)
        mouth=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE); face=cv2.imread(str(fp),cv2.IMREAD_GRAYSCALE)
        if mouth is None or face is None: continue
        if mouth.shape!=(MOUTH_SIZE,MOUTH_SIZE): mouth=cv2.resize(mouth,(MOUTH_SIZE,MOUTH_SIZE),interpolation=cv2.INTER_AREA)
        if face.shape!=(FACE_SIZE,FACE_SIZE): face=cv2.resize(face,(FACE_SIZE,FACE_SIZE),interpolation=cv2.INTER_AREA)
        mouths.append(mouth.astype(np.uint8,copy=False)); faces.append(face.astype(np.uint8,copy=False)); indices.append(fi)
    if not faces:return 'empty'
    np.savez(dst,face=np.stack(faces),mouth=np.stack(mouths),frame_indices=np.asarray(indices,dtype=np.int32))
    return 'saved'
all_df=pd.concat([train_df,val_df,test_df],ignore_index=True).drop_duplicates('clip_id')
stats={'saved':0,'skip':0,'empty':0}; t0=time.time()
for i,(_,row) in enumerate(all_df.iterrows(),1):
    s=build_one_cache(row); stats[s]+=1
    if i==1 or i%50==0 or i==len(all_df): print(f'{i:4d}/{len(all_df)} | {stats} | {(time.time()-t0)/60:.1f} min')
print('CACHE BUILD FINISHED:',stats)

In [ ]:
class FastDualStreamDataset(Dataset):
    def __init__(self,dataframe,train=False): self.df=dataframe.reset_index(drop=True).copy(); self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; cp=clip_cache_path(row['clip_id'])
        if not cp.exists(): raise FileNotFoundError(cp)
        with np.load(cp,allow_pickle=False) as z: face=z['face'].copy(); mouth=z['mouth'].copy()
        if self.train and random.random()<0.35:
            alpha=random.uniform(0.92,1.08); beta=random.uniform(-6,6)
            face=np.clip(face.astype(np.float32)*alpha+beta,0,255).astype(np.uint8); mouth=np.clip(mouth.astype(np.float32)*alpha+beta,0,255).astype(np.uint8)
        face=torch.from_numpy(face.astype(np.float32)/255.0).unsqueeze(1); mouth=torch.from_numpy(mouth.astype(np.float32)/255.0).unsqueeze(1)
        target=torch.tensor(encode_text(row['text']),dtype=torch.long)
        return {'face':face,'mouth':mouth,'target':target,'text':row['text'],'clip_id':row['clip_id']}
def collate_fn(batch):
    lengths=torch.tensor([x['face'].shape[0] for x in batch],dtype=torch.long); target_lengths=torch.tensor([x['target'].numel() for x in batch],dtype=torch.long)
    max_t=int(lengths.max()); B=len(batch)
    face=torch.zeros(B,max_t,1,FACE_SIZE,FACE_SIZE); mouth=torch.zeros(B,max_t,1,MOUTH_SIZE,MOUTH_SIZE)
    for i,x in enumerate(batch):
        t=x['face'].shape[0]; face[i,:t]=x['face']; mouth[i,:t]=x['mouth']
    return {'face':face,'mouth':mouth,'video_lengths':lengths,'targets':torch.cat([x['target'] for x in batch]),'target_lengths':target_lengths,'texts':[x['text'] for x in batch],'clip_ids':[x['clip_id'] for x in batch]}

In [ ]:
train_dataset=FastDualStreamDataset(train_df,True); val_dataset=FastDualStreamDataset(val_df,False); test_dataset=FastDualStreamDataset(test_df,False)
common=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate_fn)
train_loader=DataLoader(train_dataset,shuffle=True,**common); val_loader=DataLoader(val_dataset,shuffle=False,**common); test_loader=DataLoader(test_dataset,shuffle=False,**common)
print('Train batches:',len(train_loader),'Val:',len(val_loader),'Test:',len(test_loader))
t0=time.time(); batch=next(iter(train_loader)); print('First REAL cached batch:',round(time.time()-t0,2),'sec'); print(batch['face'].shape,batch['mouth'].shape,batch['texts'][:2])

In [ ]:
class SmallVisualEncoder(nn.Module):
    def __init__(self,in_ch=1,out_dim=128):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(in_ch,24,5,padding=2),nn.GroupNorm(6,24),nn.SiLU(inplace=True),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.GroupNorm(8,48),nn.SiLU(inplace=True),nn.MaxPool2d(2),nn.Conv2d(48,96,3,padding=1),nn.GroupNorm(12,96),nn.SiLU(inplace=True),nn.AdaptiveAvgPool2d((1,1))); self.proj=nn.Linear(96,out_dim)
    def forward(self,x): return self.proj(self.net(x).flatten(1))
class DualStreamLipReadingV3(nn.Module):
    def __init__(self,vocab_size,feature_dim=128,hidden_dim=192,gru_layers=2,dropout=0.10):
        super().__init__(); self.face_encoder=SmallVisualEncoder(1,feature_dim); self.mouth_encoder=SmallVisualEncoder(1,feature_dim); self.fusion=nn.Sequential(nn.Linear(feature_dim*2,feature_dim),nn.SiLU(),nn.Dropout(dropout)); self.fusion_norm=nn.LayerNorm(feature_dim); self.temporal=nn.GRU(feature_dim,hidden_dim,num_layers=gru_layers,batch_first=True,bidirectional=True,dropout=dropout if gru_layers>1 else 0.0); self.dropout=nn.Dropout(dropout); self.classifier=nn.Linear(hidden_dim*2,vocab_size)
        with torch.no_grad(): self.classifier.bias.zero_(); self.classifier.bias[BLANK_ID]=-0.75
    def forward(self,face,mouth,lengths):
        B,T=face.shape[:2]; f=self.face_encoder(face.reshape(B*T,1,FACE_SIZE,FACE_SIZE)); m=self.mouth_encoder(mouth.reshape(B*T,1,MOUTH_SIZE,MOUTH_SIZE)); fused=self.fusion(torch.cat([f,m],dim=-1)); x=self.fusion_norm(fused+m).reshape(B,T,-1); packed=nn.utils.rnn.pack_padded_sequence(x,lengths.cpu(),batch_first=True,enforce_sorted=False); packed_out,_=self.temporal(packed); out,_=nn.utils.rnn.pad_packed_sequence(packed_out,batch_first=True,total_length=T); return self.classifier(self.dropout(out))
model=DualStreamLipReadingV3(VOCAB_SIZE,FEATURE_DIM,GRU_HIDDEN,GRU_LAYERS,DROPOUT).to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()),'Device:',next(model.parameters()).device)

In [ ]:
criterion=nn.CTCLoss(blank=BLANK_ID,reduction='mean',zero_infinity=True)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=2,min_lr=1e-6)
scaler=torch.amp.GradScaler('cuda')
def blank_penalty_for_epoch(epoch):
    progress=min(max((epoch-1)/max(BLANK_PENALTY_DECAY_EPOCHS-1,1),0.0),1.0); return INITIAL_BLANK_LOGIT_PENALTY*(1.0-progress)
def apply_blank_logit_penalty(logits,penalty):
    if penalty<=0:return logits
    out=logits.clone(); out[...,BLANK_ID]-=penalty; return out
def anti_blank_regularizer(logits,input_lengths,target_lengths):
    probs=logits.float().softmax(dim=-1); bp=probs[...,BLANK_ID]; penalties=[]
    for b in range(logits.size(0)):
        T=int(input_lengths[b]); L=int(target_lengths[b]);
        if T<=0: continue
        observed=bp[b,:T].mean(); expected=1.0-min(L/max(T,1),0.95); max_blank=min(expected+BLANK_MARGIN,0.95); penalties.append(F.relu(observed-max_blank).pow(2))
    return torch.stack(penalties).mean() if penalties else logits.new_tensor(0.0)

In [ ]:
def edit_distance(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]
def corpus_cer(refs,preds): return sum(edit_distance(r,p) for r,p in zip(refs,preds))/max(sum(len(r) for r in refs),1)

In [ ]:
def train_one_epoch(model,loader,epoch_num):
    model.train(); optimizer.zero_grad(set_to_none=True); losses=[]; ctc_losses=[]; reg_losses=[]; penalty=blank_penalty_for_epoch(epoch_num); start=time.time()
    for step,batch in enumerate(loader,1):
        face=batch['face'].to(DEVICE,non_blocking=True); mouth=batch['mouth'].to(DEVICE,non_blocking=True); lengths=batch['video_lengths']; targets=batch['targets'].to(DEVICE,non_blocking=True); target_lengths=batch['target_lengths']
        with torch.amp.autocast('cuda',dtype=torch.float16):
            raw_logits=model(face,mouth,lengths); adjusted=apply_blank_logit_penalty(raw_logits,penalty); ctc_loss=criterion(adjusted.log_softmax(dim=-1).transpose(0,1),targets,lengths,target_lengths); blank_reg=anti_blank_regularizer(raw_logits,lengths,target_lengths); loss=ctc_loss+BLANK_REG_WEIGHT*blank_reg; scaled=loss/GRAD_ACCUM_STEPS
        scaler.scale(scaled).backward()
        if step%GRAD_ACCUM_STEPS==0 or step==len(loader): scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.detach().cpu())); ctc_losses.append(float(ctc_loss.detach().cpu())); reg_losses.append(float(blank_reg.detach().cpu()))
        if step==1 or step%10==0 or step==len(loader):
            elapsed=time.time()-start; eta=(elapsed/step)*(len(loader)-step)/60; print(f'train {step:4d}/{len(loader)} | loss={np.mean(losses):.4f} | ctc={np.mean(ctc_losses):.4f} | blank_reg={np.mean(reg_losses):.4f} | blank_pen={penalty:.2f} | ETA={eta:.1f}m')
    return float(np.mean(losses))
@torch.no_grad()
def evaluate(model,loader):
    model.eval(); losses=[]; preds=[]; refs=[]; blank_steps=0; total_steps=0
    for batch in loader:
        face=batch['face'].to(DEVICE,non_blocking=True); mouth=batch['mouth'].to(DEVICE,non_blocking=True); lengths=batch['video_lengths']; targets=batch['targets'].to(DEVICE,non_blocking=True); target_lengths=batch['target_lengths']
        with torch.amp.autocast('cuda',dtype=torch.float16): logits=model(face,mouth,lengths); loss=criterion(logits.log_softmax(dim=-1).transpose(0,1),targets,lengths,target_lengths)
        losses.append(float(loss.detach().cpu())); ids=logits.argmax(dim=-1).cpu()
        for b in range(ids.size(0)):
            T=int(lengths[b]); seq=ids[b,:T]; blank_steps+=int((seq==BLANK_ID).sum()); total_steps+=T
        preds.extend(ctc_greedy_decode(logits.float().cpu(),lengths)); refs.extend(batch['texts'])
    return float(np.mean(losses)),float(corpus_cer(refs,preds)),float(blank_steps/max(total_steps,1)),preds,refs

In [ ]:
print('='*80); print('REAL PREFLIGHT'); print('='*80)
batch=next(iter(train_loader)); face=batch['face'].to(DEVICE); mouth=batch['mouth'].to(DEVICE); lengths=batch['video_lengths']; targets=batch['targets'].to(DEVICE); target_lengths=batch['target_lengths']
model.train(); optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast('cuda',dtype=torch.float16):
    logits=model(face,mouth,lengths); adjusted=apply_blank_logit_penalty(logits,INITIAL_BLANK_LOGIT_PENALTY); ctc=criterion(adjusted.log_softmax(dim=-1).transpose(0,1),targets,lengths,target_lengths); reg=anti_blank_regularizer(logits,lengths,target_lengths); total=ctc+BLANK_REG_WEIGHT*reg
print('Face:',face.shape,'Mouth:',mouth.shape,'Logits:',logits.shape,'CTC:',float(ctc),'Reg:',float(reg),'Total:',float(total)); scaler.scale(total).backward(); optimizer.zero_grad(set_to_none=True); print('✅ REAL PREFLIGHT PASS')

In [ ]:
BEST_PATH=MODEL_ROOT/'best_model.pt'; LAST_PATH=MODEL_ROOT/'last_model.pt'; HISTORY_PATH=MODEL_ROOT/'training_history.csv'
history=[]; start_epoch=1; best_score=float('inf'); bad_epochs=0
if RESUME_IF_AVAILABLE and LAST_PATH.exists():
    ckpt=torch.load(LAST_PATH,map_location=DEVICE,weights_only=False); model.load_state_dict(ckpt['model_state_dict']);
    if 'optimizer_state_dict' in ckpt: optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt: scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch=int(ckpt.get('epoch',0))+1; best_score=float(ckpt.get('best_score',float('inf'))); bad_epochs=int(ckpt.get('bad_epochs',0));
    if HISTORY_PATH.exists(): history=pd.read_csv(HISTORY_PATH).to_dict('records')
    print('✅ Resume from epoch:',start_epoch)
else: print('Start fresh V3')
print('='*90); print('START DUAL-STREAM V3 ANTI-BLANK TRAINING'); print('='*90)
for epoch in range(start_epoch,EPOCHS+1):
    t0=time.time(); print(f'\n🚀 Epoch {epoch}/{EPOCHS}'); train_loss=train_one_epoch(model,train_loader,epoch); val_loss,val_cer,val_blank,val_preds,val_refs=evaluate(model,val_loader); scheduler.step(val_loss); lr_now=optimizer.param_groups[0]['lr']; mins=(time.time()-t0)/60; selection_score=val_cer+0.25*val_blank
    history.append({'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss,'val_cer':val_cer,'val_blank_ratio':val_blank,'selection_score':selection_score,'blank_logit_penalty':blank_penalty_for_epoch(epoch),'lr':lr_now,'minutes':mins}); pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
    print(f'\nEpoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | CER={val_cer:.4f} | blank={val_blank*100:.1f}% | score={selection_score:.4f} | lr={lr_now:.2e} | {mins:.1f} min')
    for r,p in list(zip(val_refs,val_preds))[:3]: print('REF :',r); print('PRED:',p)
    save_obj={'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),'scaler_state_dict':scaler.state_dict(),'epoch':epoch,'best_score':best_score,'bad_epochs':bad_epochs,'val_loss':val_loss,'val_cer':val_cer,'val_blank_ratio':val_blank,'vocab_size':VOCAB_SIZE,'itos':itos}; torch.save(save_obj,LAST_PATH)
    if selection_score<best_score:
        best_score=selection_score; bad_epochs=0; save_obj['best_score']=best_score; save_obj['bad_epochs']=bad_epochs; torch.save(save_obj,BEST_PATH); print('✅ Saved BEST:',BEST_PATH)
    else: bad_epochs+=1
    save_obj['best_score']=best_score; save_obj['bad_epochs']=bad_epochs; torch.save(save_obj,LAST_PATH)
    if epoch>=5 and val_blank>=0.99 and val_cer>=0.99: print('⚠️ WARNING: blank collapse ยังหนักหลังอย่างน้อย 5 epochs')
    if bad_epochs>=PATIENCE: print(f'Early stopping: {PATIENCE} epochs ไม่มี improvement'); break
print('TRAINING FINISHED')

In [ ]:
if not BEST_PATH.exists(): raise FileNotFoundError(BEST_PATH)
ckpt=torch.load(BEST_PATH,map_location=DEVICE,weights_only=False); model.load_state_dict(ckpt['model_state_dict'])
test_loss,test_cer,test_blank,test_preds,test_refs=evaluate(model,test_loader)
print('='*80); print('FINAL TEST — DUAL STREAM V3'); print('='*80); print('Best epoch:',ckpt.get('epoch')); print(f'Test Loss: {test_loss:.4f}'); print(f'Test CER: {test_cer:.4f}'); print(f'Blank Ratio: {test_blank:.4f} ({test_blank*100:.2f}%)')
for i,(r,p) in enumerate(zip(test_refs,test_preds)):
    print(f'\n[{i:03d}]'); print('REF :',r); print('PRED:',p)
    if i>=9: break

In [ ]:
config={'version':'v3_dualstream_anti_blank_fast_cache','architecture':'FaceCNN + MouthCNN + MouthResidualFusion + BiGRU + CTC','face_size':FACE_SIZE,'mouth_size':MOUTH_SIZE,'max_frames':MAX_FRAMES,'batch_size':BATCH_SIZE,'feature_dim':FEATURE_DIM,'gru_hidden':GRU_HIDDEN,'gru_layers':GRU_LAYERS,'dropout':DROPOUT,'learning_rate':LR,'blank_id':BLANK_ID,'vocab_size':VOCAB_SIZE,'blank_reg_weight':BLANK_REG_WEIGHT,'initial_blank_logit_penalty':INITIAL_BLANK_LOGIT_PENALTY,'blank_penalty_decay_epochs':BLANK_PENALTY_DECAY_EPOCHS,'best_metric':'CER + 0.25 * blank_ratio'}
with open(MODEL_ROOT/'model_config.json','w',encoding='utf-8') as f: json.dump(config,f,ensure_ascii=False,indent=2)
print('Saved:',MODEL_ROOT/'model_config.json')

## วิธีรัน
Run Cell 1 → จบ ตามลำดับ

ดู Epoch 1–5 เป็นหลัก: `CER`, `blank`, `PRED`.
ถ้าหลัง Epoch 5–8 ยัง `blank≈100%` และ `CER≈1.0` ให้หยุดก่อน ไม่ต้องรอครบ 30 epochs.